# s08 — WX, Moran bivariado y diagnóstico OLS
## CM0866 · Clase 8 · EAFIT 2026

---

| Bloque | Contenido | Tiempo |
|---|---|---|
| **A** | WX · Moran bivariado · LISA bivariado — **Ciclistas Medellín** | 35 min |
| **B** | WX · Moran bivariado · LISA bivariado — **Brexit UK** | 25 min |
| **C** | OLS + Moran I sobre residuos — **Ciclistas** | 15 min |
| **D** | OLS + Moran I sobre residuos — **Brexit** | 10 min |
| **E** | Datos propios del equipo | 30 min |

> *La receta es siempre la misma: WX → Moran bivariado global → LISA bivariado → OLS → I(ε̂)*
> *Primero ciclistas completo, luego Brexit completo.*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 0 — Librerías y datos
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from libpysal.weights import KNN
from libpysal import weights
import esda
import splot.esda as esda_plot
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family':'sans-serif','figure.facecolor':'white'})
os.makedirs('outputs', exist_ok=True)
np.random.seed(42)

# ── Ciclistas ────────────────────────────────────────────────────────────────
ciclistas = gpd.read_file('data/Ospina_cyclists_origins_4326_BD.shp').to_crs('EPSG:32618')
for v in ['length_km','o_cbd','stratum']:
    ciclistas[v] = pd.to_numeric(ciclistas[v], errors='coerce')
w_cic = KNN.from_dataframe(ciclistas, k=5)
w_cic.transform = 'r'

# ── Brexit ───────────────────────────────────────────────────────────────────
ref  = pd.read_csv('data/brexit_vote.csv')
lads = gpd.read_file('data/local_authority_districts.geojson')
db   = lads.merge(
    ref[['Area_Code','Pct_Leave','Pct_Turnout']],
    left_on='lad16cd', right_on='Area_Code'
).to_crs(epsg=3857).dropna(subset=['Pct_Leave'])
w_brex = KNN.from_dataframe(db, k=8)
w_brex.transform = 'R'

col_lisa = {1:'#2563EB',2:'#16A34A',3:'#DC2626',4:'#F97316',0:'#E2E8F0'}
nom_lisa  = {1:'HH',2:'LH',3:'LL',4:'HL',0:'No significativo'}

print(f'Ciclistas: {len(ciclistas)} · Brexit: {len(db)} distritos ✓')

---
## Bloque A — Ciclistas Medellín

**Variable dependiente:** `length_km` — distancia del viaje

**Variables explicativas candidatas:** `o_cbd` (distancia al CBD), `stratum` (estrato)

**Secuencia:**
1. WX — calcular el rezago espacial de X
2. Mapas: Y · WY · X · WX
3. Moran I bivariado global — I_yx
4. Scatter plot bivariado
5. LISA bivariado — mapa de co-distribución significativa

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA A1 — WX ciclistas: misma operación, diferente variable
# ─────────────────────────────────────────────────────────────────────────────

y_cic  = ciclistas['length_km'].values
x_cbd  = ciclistas['o_cbd'].values
x_str  = ciclistas['stratum'].values

Wy_cic  = weights.lag_spatial(w_cic, y_cic)
Wx_cbd  = weights.lag_spatial(w_cic, x_cbd)
Wx_str  = weights.lag_spatial(w_cic, x_str)

ciclistas['Wy']     = Wy_cic
ciclistas['Wx_cbd'] = Wx_cbd

print('─── Para cada ciclista i ─────────────────────────────────────────────────')
print('  y_i      = distancia de su propio viaje')
print('  Wy_i     = distancia promedio de los 5 vecinos más cercanos')
print('  x_cbd_i  = distancia de su origen al CBD')
print('  Wx_cbd_i = distancia promedio al CBD de los 5 orígenes vecinos')
print()
print(f'  ȳ={y_cic.mean():.2f} km · Wy̅={Wy_cic.mean():.2f} km')
print(f'  x̄_cbd={x_cbd.mean():.2f} · Wx̄_cbd={Wx_cbd.mean():.2f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA A2 — Mapas: Y · WY · X · WX
# La misma W transporta cosas distintas
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
configs = [
    ('length_km', 'YlOrRd', 'Y — distancia propia\n(¿cuánto viaja cada ciclista?)'),
    ('Wy',        'YlOrRd', 'WY — distancia del entorno\n(¿cuánto viajan sus vecinos?)'),
    ('o_cbd',     'Blues',  'X — distancia al CBD\n(¿qué tan lejos está del centro?)'),
    ('Wx_cbd',    'Blues',  'WX — CBD del entorno\n(¿qué tan lejos están sus vecinos del centro?)'),
]
for ax, (col, cmap, titulo) in zip(axes, configs):
    ciclistas.plot(column=col, cmap=cmap, ax=ax, markersize=4, alpha=0.7,
                   legend=True,
                   legend_kwds={'shrink':0.5,'orientation':'horizontal','pad':0.02})
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_axis_off()
plt.suptitle('Y, WY, X y WX — la misma W, diferente información\nCiclistas Medellín · KNN k=5',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_A2_wy_wx_mapas_cic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA A3 — Moran I bivariado global: ciclistas
# I_yx = r(z_y, Wz_x) · asimetría I_yx ≠ I_xy
# ─────────────────────────────────────────────────────────────────────────────

pares_cic = [
    ('length_km', 'o_cbd',   'distancia al CBD'),
    ('length_km', 'stratum', 'estrato del origen'),
]

print('─── Moran I bivariado global — Ciclistas · KNN k=5 ─────────────────────')
print(f'  {"Y vs. W·X":<30} {"I_yx":>8} {"p_yx":>8} {"I_xy":>8} {"p_xy":>8}')
print('  ' + '─'*68)

resultados_bv = {}
for y_nom, x_nom, etiqueta in pares_cic:
    y = ciclistas[y_nom].values
    x = ciclistas[x_nom].values
    mask = ~(np.isnan(y)|np.isnan(x))
    m_yx = esda.Moran_BV(y[mask], x[mask], w_cic, permutations=999)
    m_xy = esda.Moran_BV(x[mask], y[mask], w_cic, permutations=999)
    resultados_bv[x_nom] = m_yx
    sig = '✓' if m_yx.p_sim < 0.05 else '✗'
    print(f'  {y_nom} vs W·{x_nom:<18} '
          f'{m_yx.I:>8.4f} {m_yx.p_sim:>8.4f} '
          f'{m_xy.I:>8.4f} {m_xy.p_sim:>8.4f}  {sig}')
print()
print('─── Interpretación ──────────────────────────────────────────────────────')
print('  I(length_km, W·o_cbd) = 0.43 sig. → co-distribución espacial real')
print('  Los ciclistas que viajan lejos parten de zonas rodeadas de orígenes')
print('  también alejados del CBD.')
print()
print('  I(length_km, W·stratum) ≈ 0, no sig. → sin co-distribución con estrato')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA A4 — Scatter plot bivariado: ciclistas
# Eje X: z_y · Eje Y: Wz_x · Pendiente: I_yx
# ─────────────────────────────────────────────────────────────────────────────

import splot.esda as esda_plot

m_bv_cbd = resultados_bv['o_cbd']
m_bv_str = resultados_bv['stratum']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

esda_plot.moran_scatterplot(m_bv_cbd, ax=axes[0])
axes[0].set_title(f'length_km vs. W·o_cbd\nI_yx={m_bv_cbd.I:.4f} · p={m_bv_cbd.p_sim:.4f}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('z_y  (length_km − ȳ) / s', fontsize=10)
axes[0].set_ylabel('Wz_x  (rezago de o_cbd)', fontsize=10)

esda_plot.moran_scatterplot(m_bv_str, ax=axes[1])
axes[1].set_title(f'length_km vs. W·stratum\nI_yx={m_bv_str.I:.4f} · p={m_bv_str.p_sim:.4f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('z_y  (length_km − ȳ) / s', fontsize=10)
axes[1].set_ylabel('Wz_x  (rezago de stratum)', fontsize=10)

plt.suptitle('Scatter plots bivariados — Ciclistas Medellín\n'
             'o_cbd: co-distribución significativa · stratum: sin co-distribución',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_A4_scatter_bv_cic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA A5 — LISA bivariado: ciclistas
# length_km vs. W·o_cbd · mapa de co-distribución significativa
# ─────────────────────────────────────────────────────────────────────────────

ml_bv_cic  = esda.Moran_Local_BV(y_cic, x_cbd, w_cic, permutations=999)
sig_bv_cic = ml_bv_cic.p_sim < 0.05
ciclistas['lisa_bv']  = ml_bv_cic.q * sig_bv_cic.astype(int)

# LISA univariado para comparar
ml_uni_cic = esda.Moran_Local(y_cic, w_cic, permutations=999)
sig_uni    = ml_uni_cic.p_sim < 0.05
ciclistas['lisa_uni'] = ml_uni_cic.q * sig_uni.astype(int)

col_lisa = {1:'#2563EB',2:'#16A34A',3:'#DC2626',4:'#F97316',0:'#E2E8F0'}
nom_lisa  = {1:'HH',2:'LH',3:'LL',4:'HL',0:'No sig.'}
nombres_q = {1:'HH',2:'LH',3:'LL',4:'HL'}

print(f'LISA bivariado (length_km vs W·o_cbd): {sig_bv_cic.sum()} sig. de {len(ciclistas)}')
for q,n in nombres_q.items():
    s=((ml_bv_cic.q==q)&sig_bv_cic).sum()
    if s>0: print(f'  {n}: {s}')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Panel izquierdo: LISA univariado
ax1 = axes[0]
ciclistas.plot(ax=ax1, color='#E2E8F0', markersize=3, edgecolor='none', zorder=1)
for q, color in col_lisa.items():
    if q==0: continue
    mask = ciclistas['lisa_uni']==q
    if mask.sum()>0:
        ciclistas[mask].plot(ax=ax1, color=color, markersize=5, alpha=0.8, zorder=3)
import matplotlib.patches as mpatches
ax1.legend(handles=[mpatches.Patch(color=col_lisa[q],label=f'{nom_lisa[q]} — length_km')
           for q in [1,3,4,2,0]], loc='lower right', fontsize=8)
ax1.set_title(f'LISA univariado\nlength_km vs. W·length_km\n{sig_uni.sum()} sig.',
              fontsize=10, fontweight='bold')
ax1.set_axis_off()

# Panel derecho: LISA bivariado
ax2 = axes[1]
ciclistas.plot(ax=ax2, color='#E2E8F0', markersize=3, edgecolor='none', zorder=1)
for q, color in col_lisa.items():
    if q==0: continue
    mask = ciclistas['lisa_bv']==q
    if mask.sum()>0:
        ciclistas[mask].plot(ax=ax2, color=color, markersize=5, alpha=0.8, zorder=3)
ax2.legend(handles=[mpatches.Patch(color=col_lisa[q],label=f'{nom_lisa[q]} — length_km/o_cbd')
           for q in [1,3,4,2,0]], loc='lower right', fontsize=8)
ax2.set_title(f'LISA bivariado\nlength_km vs. W·o_cbd\n{sig_bv_cic.sum()} sig.',
              fontsize=10, fontweight='bold')
ax2.set_axis_off()

plt.suptitle('Ciclistas Medellín · KNN k=5\n'
             'LISA univariado: ¿dónde se agrupan? → LISA bivariado: ¿dónde coexisten con el contexto de X?',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_A5_lisa_bv_cic.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Bloque B — Brexit UK

**Variable dependiente:** `Pct_Leave` — porcentaje de voto Leave

**Variable explicativa candidata:** `Pct_Turnout` — participación electoral

**Misma secuencia:** WX → mapas → Moran bivariado global → scatter → LISA bivariado

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA B1 — WX Brexit
# ─────────────────────────────────────────────────────────────────────────────

y_brex  = db['Pct_Leave'].values
x_turn  = db['Pct_Turnout'].values

Wy_brex = weights.lag_spatial(w_brex, y_brex)
Wx_turn = weights.lag_spatial(w_brex, x_turn)

db['Wy']      = Wy_brex
db['Wx_turn'] = Wx_turn

print('─── Para cada distrito i ────────────────────────────────────────────────')
print('  y_i       = % voto Leave propio')
print('  Wy_i      = % Leave promedio de los 8 distritos vecinos')
print('  x_turn_i  = % participación electoral propia')
print('  Wx_turn_i = % participación promedio de los 8 distritos vecinos')
print()
print(f'  ȳ_Leave={y_brex.mean():.2f}%  Wy̅_Leave={Wy_brex.mean():.2f}%')
print(f'  x̄_Turn={x_turn.mean():.2f}%  Wx̄_Turn={Wx_turn.mean():.2f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA B2 — Mapas: Y · WY · X · WX — Brexit
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 7))
configs_b = [
    ('Pct_Leave', 'RdBu_r', 'Y — % voto Leave propio'),
    ('Wy',        'RdBu_r', 'WY — % Leave del entorno'),
    ('Pct_Turnout','YlOrRd','X — % participación propia'),
    ('Wx_turn',   'YlOrRd', 'WX — % participación del entorno'),
]
for ax, (col, cmap, titulo) in zip(axes, configs_b):
    db.plot(column=col, cmap=cmap, ax=ax,
            edgecolor='white', linewidth=0.15,
            legend=True,
            legend_kwds={'shrink':0.5,'orientation':'horizontal','pad':0.02})
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_axis_off()
plt.suptitle('Y, WY, X y WX — Brexit UK · KNN k=8\n'
             'La misma W transporta el resultado y las características del entorno',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_B2_wy_wx_mapas_brex.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA B3 — Moran I bivariado global + scatter: Brexit
# ─────────────────────────────────────────────────────────────────────────────

m_yx_b = esda.Moran_BV(y_brex, x_turn, w_brex, permutations=999)
m_xy_b = esda.Moran_BV(x_turn, y_brex, w_brex, permutations=999)

print('─── Moran I bivariado global — Brexit · KNN k=8 ─────────────────────────')
print(f'  I(Leave, W·Turnout) = {m_yx_b.I:.4f}  p = {m_yx_b.p_sim:.4f}')
print(f'  I(Turnout, W·Leave) = {m_xy_b.I:.4f}  p = {m_xy_b.p_sim:.4f}')
print(f'  Asimetría: {abs(m_yx_b.I - m_xy_b.I):.4f}')
print()
print('  Los distritos con alto Leave tienden a estar rodeados de distritos')
print('  con alta participación — la geografía del voto y la participación coexisten.')

fig, ax = plt.subplots(figsize=(8, 7))
esda_plot.moran_scatterplot(m_yx_b, ax=ax)
ax.set_title(f'Moran Scatterplot bivariado — Brexit UK\n'
             f'Pct_Leave vs. W·Pct_Turnout\n'
             f'I_yx = {m_yx_b.I:.4f} · p = {m_yx_b.p_sim:.4f}',
             fontsize=12, fontweight='bold')
ax.set_xlabel('z_y  (Pct_Leave − ȳ) / s', fontsize=11)
ax.set_ylabel('Wz_x  (rezago de Pct_Turnout)', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/s08_B3_scatter_bv_brex.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA B4 — LISA bivariado: Brexit
# Pct_Leave vs. W·Pct_Turnout · mapa de co-distribución significativa
# ─────────────────────────────────────────────────────────────────────────────

ml_bv_brex  = esda.Moran_Local_BV(y_brex, x_turn, w_brex, permutations=999)
sig_bv_brex = ml_bv_brex.p_sim < 0.05
db['lisa_bv']  = ml_bv_brex.q * sig_bv_brex.astype(int)

ml_uni_brex  = esda.Moran_Local(y_brex, w_brex, permutations=999)
sig_uni_brex = ml_uni_brex.p_sim < 0.05
db['lisa_uni'] = ml_uni_brex.q * sig_uni_brex.astype(int)

print(f'LISA bivariado (Leave vs W·Turnout): {sig_bv_brex.sum()} sig. de {len(db)}')
for q,n in nombres_q.items():
    s=((ml_bv_brex.q==q)&sig_bv_brex).sum()
    if s>0: print(f'  {n}: {s}')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, col_campo, titulo, n_sig in [
    (axes[0],'lisa_uni',
     f'LISA univariado\nPct_Leave vs. W·Pct_Leave\n{sig_uni_brex.sum()} sig.',
     sig_uni_brex.sum()),
    (axes[1],'lisa_bv',
     f'LISA bivariado\nPct_Leave vs. W·Pct_Turnout\n{sig_bv_brex.sum()} sig.',
     sig_bv_brex.sum()),
]:
    db.plot(ax=ax, color='#E2E8F0', edgecolor='white', linewidth=0.15, zorder=1)
    for q, color in col_lisa.items():
        if q==0: continue
        mask = db[col_campo]==q
        if mask.sum()>0:
            db[mask].plot(ax=ax, color=color, edgecolor='white',
                          linewidth=0.15, zorder=3)
    ax.legend(handles=[mpatches.Patch(color=col_lisa[q],label=nom_lisa[q])
                       for q in [1,3,4,2,0]], loc='lower left', fontsize=8)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_axis_off()

plt.suptitle('Brexit UK · KNN k=8\n'
             'LISA univariado: ¿dónde se agrupa el voto? → LISA bivariado: ¿dónde coexiste con participación alta?',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_B4_lisa_bv_brex.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Bloque C — OLS y Moran I sobre residuos: Ciclistas

Pregunta: *¿el modelo OLS capturó la dependencia espacial o quedó algo sin explicar?*

Estimamos dos modelos:
- **OLS simple:** solo `o_cbd` y `stratum` — el punto de partida
- **OLS completo (Ospina et al., 2020):** las 32 variables del paper publicado

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA C1 — OLS simple vs. OLS completo · ciclistas
# ─────────────────────────────────────────────────────────────────────────────

import spreg

vars_completo = [
    'age','bHinc','bMinc','bgender','bbothpurp','bstudy',
    'bpubbike','bbothbike','frecweek','bdispo_pv','bdispo_tp',
    'bintermoda','bretour',
    'o_i_dens','o_km_dens','o_cbd','o_pLU_Hmix','o_LCycle','o_bsindex','o_Hls','o_Lls',
    'd_i_dens','d_km_dens','d_pLU_Hmix','d_LCycle','d_bsindex','d_Hls','d_Lls',
    'p_Lcycle','r_Hls','r_Lls','r_pLU_Hmix',
]
for v in vars_completo:
    ciclistas[v] = pd.to_numeric(ciclistas[v], errors='coerce')

# OLS simple
cic_s = ciclistas.dropna(subset=['length_km','o_cbd','stratum']).copy()
w_s   = KNN.from_dataframe(cic_s, k=5); w_s.transform='r'
ols_s = spreg.OLS(
    cic_s['length_km'].values.reshape(-1,1),
    np.column_stack([cic_s['o_cbd'].values, cic_s['stratum'].values]),
    w=w_s, name_y='length_km', name_x=['o_cbd','stratum'], spat_diag=True)

# OLS completo
cols_c = ['length_km','geometry'] + vars_completo
cic_c  = gpd.GeoDataFrame(ciclistas[cols_c].dropna().reset_index(drop=True),
                           geometry='geometry', crs=ciclistas.crs)
w_c    = KNN.from_dataframe(cic_c, k=5); w_c.transform='r'
ols_c  = spreg.OLS(
    cic_c['length_km'].values.reshape(-1,1),
    cic_c[vars_completo].values,
    w=w_c, name_y='length_km', name_x=vars_completo, spat_diag=True)

res_s = ols_s.u.flatten(); cic_s['residuo_ols'] = res_s
res_c = ols_c.u.flatten(); cic_c['residuo_ols'] = res_c
mi_s  = esda.Moran(res_s, w_s, permutations=999)
mi_c  = esda.Moran(res_c, w_c, permutations=999)

print('─── Comparación OLS — Ciclistas ─────────────────────────────────────────')
print(f'  {"":28} {"OLS simple":>12} {"OLS completo":>14}')
print('  ' + '─'*56)
print(f'  {"n":28} {len(cic_s):>12} {len(cic_c):>14}')
print(f'  {"R²":28} {ols_s.r2:>12.4f} {ols_c.r2:>14.4f}')
print(f'  {"Moran I(ε̂)":28} {mi_s.I:>12.4f} {mi_c.I:>14.4f}')
print(f'  {"p-valor":28} {mi_s.p_sim:>12.4f} {mi_c.p_sim:>14.4f}')
print(f'  {"RLM-lag sig.":28} {str(ols_s.rlm_lag[1]<0.05):>12} {str(ols_c.rlm_lag[1]<0.05):>14}')
print(f'  {"RLM-error sig.":28} {str(ols_s.rlm_error[1]<0.05):>12} {str(ols_c.rlm_error[1]<0.05):>14}')
print(f'  {"Modelo sugerido":28} {"SAR":>12} {"SEM":>14}')
print()
print('─── Revelación ──────────────────────────────────────────────────────────')
print('  El OLS completo (R²=0.84) tiene un Moran I de residuos MÁS ALTO')
print('  que el OLS simple (R²=0.21): 0.213 > 0.122')
print('  Las 32 variables no absorbieron la dependencia espacial.')
print('  El paper (Ospina et al., 2020) necesitaba un SEM.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA C2 — Mapas de residuos + Moran scatter: ciclistas
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for ax, data, res, mi, titulo in [
    (axes[0], cic_s, res_s, mi_s,
     f'OLS simple (2 vars)\nR²=0.21 · I(ε̂)={mi_s.I:.3f} · p={mi_s.p_sim:.3f}'),
    (axes[1], cic_c, res_c, mi_c,
     f'OLS completo — Ospina et al. (2020)\nR²=0.84 · I(ε̂)={mi_c.I:.3f} · p={mi_c.p_sim:.3f}'),
]:
    vmax = np.abs(res).max()
    data.plot(column='residuo_ols', cmap='RdBu_r', ax=ax,
              vmin=-vmax, vmax=vmax, markersize=4, alpha=0.8,
              legend=True,
              legend_kwds={'shrink':0.5,'orientation':'horizontal',
                           'label':'Residuo (km)','pad':0.02})
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_axis_off()

esda_plot.moran_scatterplot(mi_c, ax=axes[2])
axes[2].set_title(f'Moran Scatterplot — residuos OLS completo\n'
                  f'I(ε̂)={mi_c.I:.4f} · p={mi_c.p_sim:.3f} → SEM sugerido',
                  fontsize=10, fontweight='bold')
axes[2].set_xlabel('Residuo estandarizado', fontsize=10)
axes[2].set_ylabel('W · Residuo', fontsize=10)

plt.suptitle('Ciclistas Medellín — Un R² alto no garantiza ausencia de dependencia espacial',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_C2_residuos_cic.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Bloque D — OLS y Moran I sobre residuos: Brexit

El caso opuesto a los ciclistas:
- R² muy bajo — las variables disponibles no explican el voto
- Moran I(ε̂) muy alto — todo el patrón está en el espacio

> *La participación electoral no explica por qué unos distritos votaron Leave.*
> *El patrón es casi enteramente espacial — identidad regional, historia económica.*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA D1 — OLS Brexit + Moran I sobre residuos
# Solo Pct_Turnout — única variable explicativa disponible en el dataset
# ─────────────────────────────────────────────────────────────────────────────

import spreg

# db ya tiene Pct_Leave y Pct_Turnout del merge inicial
db_b = db.dropna(subset=['Pct_Leave','Pct_Turnout']).copy()
w_b  = KNN.from_dataframe(db_b, k=8)
w_b.transform = 'R'

y_b_ols = db_b['Pct_Leave'].values.reshape(-1, 1)
X_b_ols = db_b['Pct_Turnout'].values.reshape(-1, 1)

ols_b = spreg.OLS(
    y_b_ols, X_b_ols, w=w_b,
    name_y='Pct_Leave',
    name_x=['Pct_Turnout'],
    spat_diag=True
)
res_b = ols_b.u.flatten()
mi_b  = esda.Moran(res_b, w_b, permutations=999)
db_b['residuo_ols'] = res_b

print('─── OLS Brexit · Pct_Leave ~ Pct_Turnout ────────────────────────────────')
print(f'  n={len(db_b)}  R²={ols_b.r2:.4f}  I(ε̂)={mi_b.I:.4f}  p={mi_b.p_sim:.4f}')
print()
print('─── Tests LM ────────────────────────────────────────────────────────────')
print(f'  LM-lag:    {ols_b.lm_lag[0]:.4f}   p={ols_b.lm_lag[1]:.4f}')
print(f'  LM-error:  {ols_b.lm_error[0]:.4f}   p={ols_b.lm_error[1]:.4f}')
print(f'  RLM-lag:   {ols_b.rlm_lag[0]:.4f}   p={ols_b.rlm_lag[1]:.4f}')
print(f'  RLM-error: {ols_b.rlm_error[0]:.4f}   p={ols_b.rlm_error[1]:.4f}')
print()
print('─── Árbol de decisión ───────────────────────────────────────────────────')
rl = ols_b.rlm_lag[1]   < 0.05
re = ols_b.rlm_error[1] < 0.05
if rl and re:
    print('  Ambos RLM sig. → SDM o comparar magnitudes')
elif rl:
    print('  RLM-lag sig., RLM-error no → SAR')
    print('  El voto Leave se contagia entre distritos vecinos')
elif re:
    print('  RLM-error sig., RLM-lag no → SEM')
else:
    print('  Ningún RLM sig. → OLS suficiente')
print()
print('─── Comparación final ───────────────────────────────────────────────────')
print(f'  {"":28} {"Ciclistas":>12} {"Brexit":>10}')
print('  ' + '─'*52)
print(f'  {"R² del OLS":28} {ols_c.r2:>12.3f} {ols_b.r2:>10.3f}')
print(f'  {"Moran I(ε̂)":28} {mi_c.I:>12.3f} {mi_b.I:>10.3f}')
print(f'  {"p-valor":28} {mi_c.p_sim:>12.4f} {mi_b.p_sim:>10.4f}')
print()
print('→ El diagnóstico correcto es siempre el Moran I(ε̂) — no el R².')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA D2 — Mapa de residuos Brexit + comparación visual
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel izquierdo: residuos Brexit
vmax_b = np.abs(res_b).max()
db_b.plot(column='residuo_ols', cmap='RdBu_r', ax=axes[0],
          vmin=-vmax_b, vmax=vmax_b, edgecolor='white', linewidth=0.15,
          legend=True,
          legend_kwds={'shrink':0.5,'orientation':'horizontal',
                       'label':'Residuo (% Leave)','pad':0.02})
axes[0].set_title(f'Brexit UK — OLS (3 vars)\n'
                  f'R²={ols_b.r2:.3f} · I(ε̂)={mi_b.I:.3f} · SAR sugerido',
                  fontsize=11, fontweight='bold')
axes[0].set_axis_off()

# Panel derecho: Moran scatter de residuos Brexit
esda_plot.moran_scatterplot(mi_b, ax=axes[1])
axes[1].set_title(f'Moran Scatterplot — residuos OLS Brexit\n'
                  f'I(ε̂)={mi_b.I:.4f} · p={mi_b.p_sim:.3f} → SAR',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Residuo estandarizado', fontsize=10)
axes[1].set_ylabel('W · Residuo', fontsize=10)

plt.suptitle('Brexit UK — Prácticamente todo el patrón del voto está en el espacio\n'
             f'I(ε̂)={mi_b.I:.2f} ≈ I(Y)=0.65 — el OLS no capturó nada espacial',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/s08_D2_residuos_brex.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Parte D — Datos propios del equipo

**Tres pasos:**
1. Calcular WX para las variables X candidatas del proyecto
2. Moran I bivariado global para cada par Y–X
3. LISA bivariado para el par con mayor co-distribución

**Pregunta clave:**
> *¿Cuál de sus X candidatas tiene mayor co-distribución espacial con Y?*
> *¿El mapa LISA bivariado coincide geográficamente con el LISA univariado de la Clase 7?*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA D1 — WX + Moran bivariado + LISA bivariado: datos propios
# Modificar las 4 líneas de configuración
# ─────────────────────────────────────────────────────────────────────────────

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CONFIGURACIÓN — cada equipo modifica solo estas líneas                 │
# └─────────────────────────────────────────────────────────────────────────┘
RUTA_DATOS    = 'data/mi_proyecto.gpkg'
VARIABLE_Y    = 'mi_variable_y'
VARIABLES_X   = ['variable_x1', 'variable_x2']   # al menos dos
NOMBRE_EQUIPO = 'Equipo X'

try:
    from libpysal.weights import Queen
    gdf = gpd.read_file(RUTA_DATOS)
    if gdf.crs is None or gdf.crs.is_geographic:
        gdf = gdf.to_crs('EPSG:3116')
    for v in [VARIABLE_Y] + VARIABLES_X:
        gdf[v] = pd.to_numeric(gdf[v], errors='coerce')

    geom = gdf.geometry.geom_type.unique()[0]
    w_p  = (Queen.from_dataframe(gdf) if geom in ['Polygon','MultiPolygon']
            else KNN.from_dataframe(gdf, k=5))
    w_p.transform = 'r'

    y_p = gdf[VARIABLE_Y].values

    # ── Moran bivariado global para todas las X ───────────────────────────────
    print(f'─── {NOMBRE_EQUIPO} — Moran I bivariado global ──────────────────────')
    print(f'  {"Y vs. W·X":<30} {"I_yx":>8} {"p_yx":>8}  significativo?')
    print('  ' + '─'*58)

    mejor_x, mejor_I = None, 0
    for x_nom in VARIABLES_X:
        x_p = gdf[x_nom].values
        mask = ~(np.isnan(y_p)|np.isnan(x_p))
        m_bv = esda.Moran_BV(y_p[mask], x_p[mask], w_p, permutations=999)
        sig  = '✓' if m_bv.p_sim < 0.05 else '✗'
        print(f'  {VARIABLE_Y} vs W·{x_nom:<18} '
              f'{m_bv.I:>8.4f} {m_bv.p_sim:>8.4f}  {sig}')
        if abs(m_bv.I) > abs(mejor_I):
            mejor_x, mejor_I = x_nom, m_bv.I

    print()
    print(f'→ X con mayor co-distribución: {mejor_x} (I = {mejor_I:.4f})')

    # ── LISA bivariado para la X con mayor I ──────────────────────────────────
    x_mejor = gdf[mejor_x].values
    mask = ~(np.isnan(y_p)|np.isnan(x_mejor))
    ml_bv = esda.Moran_Local_BV(y_p[mask], x_mejor[mask], w_p, permutations=999)
    sig_bv = ml_bv.p_sim < 0.05
    gdf['lisa_bv'] = 0
    gdf.loc[mask, 'lisa_bv'] = ml_bv.q * sig_bv.astype(int)

    print(f'\nLISA bivariado ({VARIABLE_Y} vs W·{mejor_x}):')
    print(f'  {sig_bv.sum()} unidades significativas (pᵢ < 0.05)')
    for q, nom in {1:'HH',2:'LH',3:'LL',4:'HL'}.items():
        n = ((ml_bv.q==q)&sig_bv).sum()
        if n > 0: print(f'    {nom}: {n}')

    # ── Mapa LISA bivariado ───────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 9))
    gdf.plot(ax=ax, color='#E2E8F0', edgecolor='white', linewidth=0.2, zorder=1)
    for q, color in col_lisa.items():
        if q == 0: continue
        mask_q = gdf['lisa_bv'] == q
        if mask_q.sum() > 0:
            gdf[mask_q].plot(ax=ax, color=color,
                             edgecolor='white', linewidth=0.2,
                             markersize=6, zorder=3)
    ax.legend(handles=[mpatches.Patch(color=col_lisa[q], label=nom_lisa[q])
                       for q in [1,3,4,2,0]],
              loc='lower right', fontsize=9, framealpha=0.95)
    ax.set_title(f'LISA bivariado — {NOMBRE_EQUIPO}\n'
                 f'{VARIABLE_Y} vs. W·{mejor_x}\n'
                 f'{sig_bv.sum()} unidades significativas (pᵢ < 0.05)',
                 fontsize=12, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(f'outputs/s08_D_{NOMBRE_EQUIPO.replace(" ","_")}_lisa_bv.png',
                dpi=150, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'⚠ {e}')
    print('  Actualiza RUTA_DATOS, VARIABLE_Y y VARIABLES_X.')

---
## Resumen: la secuencia completa de la Clase 8

**La misma receta aplicada a dos datasets:**

```
Para cada dataset:
  1. WX = lag_spatial(w, x)           — rezago espacial de X
  2. Mapas Y · WY · X · WX            — visualizar qué transporta cada operación
  3. Moran_BV(y, x, w)                — co-distribución global Y y contexto de X
  4. Scatter plot bivariado            — pendiente = I_yx
  5. Moran_Local_BV(y, x, w)          — LISA bivariado · dónde coexisten
  6. spreg.OLS() + spat_diag=True     — el modelo sin componentes espaciales
  7. Moran I(ε̂)                      — ¿quedó dependencia espacial?
  8. Árbol LM                          — SAR / SEM / SDM
```

### El resultado comparativo

| | Ciclistas (Ospina 2020) | Brexit (Arribas) |
|---|---|---|
| **R² del OLS** | 0.84 | 0.035 |
| **Moran I(ε̂)** | 0.21 | 0.63 |
| **Modelo sugerido** | SEM | SAR |
| **Mecanismo** | Efectos de vecindario | Contagio territorial |

> *El diagnóstico correcto es siempre el Moran I(ε̂) — no el R².*
> *La Clase 9 estima SEM para ciclistas y SAR para Brexit.*

---
*CM0866 · Juan Pablo Ospina Zapata · EAFIT 2026*